In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Mahusay na simulation ng stabilizer circuits gamit ang Qiskit Aer primitives

<details>
<summary><b>Mga bersyon ng package</b></summary>

Ang code sa pahinang ito ay ginawa gamit ang mga sumusunod na kinakailangan.
Inirerekomenda namin ang paggamit ng mga bersyong ito o mas bago.

```
qiskit[all]~=2.3.0
qiskit-aer~=0.17
```
</details>

Ipinapakita ng pahinang ito kung paano gamitin ang Qiskit Aer primitives para mahusay na i-simulate ang mga stabilizer circuit, kasama na ang mga napapailalim sa Pauli noise.

Ang mga stabilizer circuit, na kilala rin bilang Clifford circuits, ay isang mahalagang limitadong klase ng mga quantum circuit na maaaring mahusay na i-simulate nang klasikal. Mayroong ilang katumbas na paraan para tukuyin ang mga stabilizer circuit. Ang isang kahulugan ay ang isang stabilizer circuit ay isang quantum circuit na binubuo lamang ng mga sumusunod na gate:

- [CX](../api/qiskit/qiskit.circuit.library.CXGate)
- [Hadamard](../api/qiskit/qiskit.circuit.library.HGate)
- [S](../api/qiskit/qiskit.circuit.library.SGate)
- [Measurement](../api/qiskit/circuit#qiskit.circuit.Measure)

Tandaan na gamit ang Hadamard at S, maaari tayong bumuo ng anumang Pauli rotation gate ([$R_x$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RXGate), [$R_y$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RYGate), at [$R_z$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RZGate)) na may anggulo sa loob ng set na ${0, \frac{\pi}{2}, \pi, \frac{3\pi}{2}}$ (hanggang sa global phase), kaya maaari nating isama ang mga gate na ito sa kahulugan.

Ang mga stabilizer circuit ay mahalaga sa pag-aaral ng quantum error correction. Ang kanilang klasikal na simulability ay nagpapakinabang din sa kanila para sa pag-verify ng output ng mga quantum computer. Halimbawa, ipagpalagay na gusto mong i-execute ang isang quantum circuit na gumagamit ng 100 qubit sa isang quantum computer. Paano mo malalaman na tama ang kilos ng quantum computer? Ang isang quantum circuit sa 100 qubit ay lagpas sa kakayahan ng brute-force classical simulation. Sa pamamagitan ng pagbabago ng iyong circuit para maging stabilizer circuit ito, maaari kang magpatakbo ng mga circuit sa quantum computer na may katulad na istraktura sa iyong nais na circuit, ngunit na maaari mong i-simulate sa isang klasikal na computer. Sa pamamagitan ng pagsusuri ng output ng quantum computer sa mga stabilizer circuit, makakakuha ka ng kumpiyansa na tama rin ang kilos nito sa mga non-stabilizer circuit. Tingnan ang [*Evidence for the utility of quantum computing before fault tolerance*](https://www.nature.com/articles/s41586-023-06096-3) para sa isang halimbawa ng ideyang ito sa pagsasanay.

Ang [Exact and noisy simulation with Qiskit Aer primitives](simulate-with-qiskit-aer) ay nagpapakita kung paano gamitin ang [Qiskit Aer](https://qiskit.org/ecosystem/aer/) para magsagawa ng exact at noisy simulation ng mga generic na quantum circuit. Isaalang-alang ang halimbawang circuit na ginamit sa artikulong iyon, isang 8-qubit circuit na itinayo gamit ang [efficient_su2](../api/qiskit/qiskit.circuit.library.efficient_su2):

In [1]:
from qiskit.circuit.library import efficient_su2

n_qubits = 8
circuit = efficient_su2(n_qubits)
circuit.draw("mpl")

<Image src="../docs/images/guides/simulate-stabilizer-circuits/extracted-outputs/2d26ac3e-2a6a-4d73-900f-470200a63154-0.svg" alt="Output of the previous code cell" />

![Output ng nakaraang code cell](../docs/images/guides/simulate-stabilizer-circuits/extracted-outputs/2d26ac3e-2a6a-4d73-900f-470200a63154-0.svg)

Gamit ang Qiskit Aer, madali naming na-simulate ang circuit na ito. Gayunpaman, ipagpalagay na itinakda natin ang bilang ng qubit sa 500:

In [2]:
n_qubits = 500
circuit = efficient_su2(n_qubits)
# don't try to draw the circuit because it's too large

Dahil ang gastos ng pag-simulate ng mga quantum circuit ay lumalaki nang exponential kasabay ng bilang ng mga qubit, ang ganitong malaking circuit ay karaniwang lalampas sa kakayahan ng kahit isang high-performance simulator tulad ng Qiskit Aer. Ang klasikal na simulation ng mga generic na quantum circuit ay nagiging hindi magagawa kapag ang bilang ng mga qubit ay lumampas sa humigit-kumulang 50 hanggang 100 qubit. Gayunpaman, tandaan na ang efficient_su2 circuit ay parameterized ng mga anggulo sa mga $R_y$ at $R_z$ gate. Kung lahat ng mga anggulong ito ay nasa loob ng set na ${0, \frac{\pi}{2}, \pi, \frac{3\pi}{2}}$, kung gayon ang circuit ay isang stabilizer circuit, at maaari itong mahusay na i-simulate!

Sa sumusunod na cell, pinapatakbo namin ang circuit gamit ang Sampler primitive na sinusuportahan ng stabilizer circuit simulator, gamit ang mga parameter na pinili nang random upang matiyak na ang circuit ay isang stabilizer circuit.

In [3]:
import numpy as np
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2 as Sampler

measured_circuit = circuit.copy()
measured_circuit.measure_all()

rng = np.random.default_rng(1234)
params = rng.choice(
    [0, np.pi / 2, np.pi, 3 * np.pi / 2],
    size=circuit.num_parameters,
)

# Initialize a Sampler backed by the stabilizer circuit simulator
exact_sampler = Sampler(
    options=dict(backend_options=dict(method="stabilizer"))
)
# The circuit needs to be transpiled to the AerSimulator target
pass_manager = generate_preset_pass_manager(
    1, AerSimulator(method="stabilizer")
)
isa_circuit = pass_manager.run(measured_circuit)
pub = (isa_circuit, params)
job = exact_sampler.run([pub])
result = job.result()
pub_result = result[0]
counts = pub_result.data.meas.get_counts()

Sinusuportahan din ng stabilizer circuit simulator ang noisy simulation, ngunit para lamang sa isang limitadong klase ng mga noise model. Partikular, ang anumang quantum noise ay dapat na nailalarawan ng isang [Pauli error](https://qiskit.org/ecosystem/aer/stubs/qiskit_aer.noise.pauli_error.html#qiskit_aer.noise.pauli_error) channel. Ang [Depolarizing error](https://qiskit.org/ecosystem/aer/stubs/qiskit_aer.noise.depolarizing_error.html) ay nasa kategoryang ito, kaya maaari rin itong i-simulate. Ang mga klasikal na noise channel tulad ng [readout error](https://qiskit.org/ecosystem/aer/stubs/qiskit_aer.noise.ReadoutError.html) ay maaari ring i-simulate.

Ang sumusunod na code cell ay nagpapatakbo ng parehong simulation tulad ng dati, ngunit sa pagkakataong ito ay tinutukoy ang isang noise model na nagdaragdag ng depolarizing error na 2% sa bawat CX gate, pati na rin ang readout error na nagbabaliktad ng bawat nasukat na bit na may posibilidad na 5%.

In [4]:
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError

noise_model = NoiseModel()
cx_depolarizing_prob = 0.02
bit_flip_prob = 0.05
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(cx_depolarizing_prob, 2), ["cx"]
)
noise_model.add_all_qubit_readout_error(
    ReadoutError(
        [
            [1 - bit_flip_prob, bit_flip_prob],
            [bit_flip_prob, 1 - bit_flip_prob],
        ]
    )
)

noisy_sampler = Sampler(
    options=dict(
        backend_options=dict(method="stabilizer", noise_model=noise_model)
    )
)
job = noisy_sampler.run([pub])
result = job.result()
pub_result = result[0]
counts = pub_result.data.meas.get_counts()

Ngayon, gamitin natin ang Estimator primitive na sinusuportahan ng stabilizer simulator para kalkulahin ang inaasahang halaga ng observable na $ZZ \cdots Z$. Dahil sa espesyal na istraktura ng mga stabilizer circuit, ang resulta ay malamang na 0.

In [5]:
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2 as Estimator

observable = SparsePauliOp("Z" * n_qubits)

exact_estimator = Estimator(
    options=dict(backend_options=dict(method="stabilizer")),
)
isa_circuit = pass_manager.run(circuit)
pub = (isa_circuit, observable, params)
job = exact_estimator.run([pub])
result = job.result()
pub_result = result[0]
exact_value = float(pub_result.data.evs)
exact_value

0.0

## Next steps

<Admonition type="tip" title="Recommendations">
  - To simulate circuits with Qiskit Aer, see [Exact and noisy simulation with Qiskit Aer primitives](./simulate-with-qiskit-sdk-primitives).
  - Review the [Qiskit Aer](https://qiskit.org/ecosystem/aer/) documentation.
</Admonition>